# Round 6: lightGBM with FE (trees and linear)

In [9]:
## ADD SAVE PARAMETERS
save = True
save_name = "round6_lightGBM_fe_trees"
#save_name = "round6_lightGBM_fe_linear-200"
notes = "lightGBM using feature-engineered features without duplication (for linear-) and 200 optuna trials"

In [10]:
# import statements
import numpy as np
import pandas as pd
import json, os
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import optuna


In [11]:
# load processed data
X_train = pd.read_csv("../data/processed/feature_engineering/X_train_fe_trees.csv")
#X_train = pd.read_csv("../data/processed/feature_engineering/X_train_fe_linear.csv")
X_test = pd.read_csv("../data/processed/feature_engineering/X_test_fe_trees.csv")
#X_test = pd.read_csv("../data/processed/feature_engineering/X_test_fe_linear.csv")
Y_train = pd.read_csv("../data/processed/Y_train.csv")

In [12]:
# ── Optuna objective ─────────────────────────────────────────

def objective(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 100, 2000, step=100),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth'         : trial.suggest_int('max_depth', 3, 12),
        'num_leaves'        : trial.suggest_int('num_leaves', 20, 300),
        'min_child_samples' : trial.suggest_int('min_child_samples', 5, 100),
        'subsample'         : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state'      : 42,
        'n_jobs'            : -1,
        'verbose'           : -1
    }

    model  = lgb.LGBMRegressor(**params)
    scores = cross_val_score(
        model, X_train, Y_train,
        cv=5,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    return -scores.mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=200, show_progress_bar=True)

print(f"\n✅ Best RMSE  : {study.best_value:.5f}")
print(f"🎛️  Best Params: {study.best_params}")

# ── Train final model ────────────────────────────────────────
lgbm = lgb.LGBMRegressor(**study.best_params, random_state=42, verbose=-1)
lgbm.fit(X_train, Y_train)

  0%|          | 0/200 [00:00<?, ?it/s]


✅ Best RMSE  : 0.12321
🎛️  Best Params: {'n_estimators': 1800, 'learning_rate': 0.008506733147896229, 'max_depth': 4, 'num_leaves': 171, 'min_child_samples': 5, 'subsample': 0.6067240176486762, 'colsample_bytree': 0.527705085392817, 'reg_alpha': 4.564567376809867e-07, 'reg_lambda': 5.246662228434935e-07}


,num_leaves,171
,max_depth,4
,learning_rate,0.008506733147896229
,n_estimators,1800
,min_child_samples,5
,subsample,0.6067240176486762
,colsample_bytree,0.527705085392817
,reg_alpha,4.564567376809867e-07
,reg_lambda,5.246662228434935e-07
,random_state,42
,verbose,-1


In [13]:
# save to output

preds = np.expm1(lgbm.predict(X_test))

submission = pd.DataFrame({
    'Id'       : pd.read_csv('../data/raw/test.csv')['Id'],
    'SalePrice': preds
})

submission.to_csv(f'../data/output/{save_name}.csv', index=False)
print(submission.head())

     Id      SalePrice
0  1461  125542.498260
1  1462  161369.975317
2  1463  173880.818868
3  1464  190304.485845
4  1465  179746.480360


In [14]:
# get cv_scores

cv_scores = cross_val_score(
    lgbm, X_train, Y_train,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

print(f"CV RMSE scores : {-cv_scores}")
print(f"Mean CV RMSE   : {-cv_scores.mean():.4f}")
print(f"Std CV RMSE    : {cv_scores.std():.4f}")

CV RMSE scores : [0.11261219 0.13519479 0.13025259 0.10818611 0.12982212]
Mean CV RMSE   : 0.1232
Std CV RMSE    : 0.0107


In [15]:
# save

log_entry = {
    "model"       : save_name,
    "cv_rmse_mean": round(float(-cv_scores.mean()), 5),
    "cv_rmse_std" : round(float(cv_scores.std()), 5),
    "cv_scores"   : [round(float(-s), 5) for s in cv_scores],
    "params"      : {},          # if using optuna, else {}
    "submission"  : f"{save_name}.csv",
    "notes"       : notes
}

if save: 
    os.makedirs('../data/output', exist_ok=True)
    log_path = '../data/output/cv_scores.json'
    # Load existing log or start fresh
    if os.path.exists(log_path):
        with open(log_path, 'r') as f:
            log = json.load(f)
    else:
        log = []

    log.append(log_entry)

    with open(log_path, 'w') as f:
        json.dump(log, f, indent=2)

    print(f"Logged CV score: {log_entry['cv_rmse_mean']:.5f} ± {log_entry['cv_rmse_std']:.5f}")
else: 
    print(f"Not logged, change to save boolean to log: {log_entry['cv_rmse_mean']:.5f} ± {log_entry['cv_rmse_std']:.5f}")

Logged CV score: 0.12321 ± 0.01072


In [16]:
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

kf  = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(X_train))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr,  X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    Y_tr,  Y_val = Y_train.iloc[tr_idx], Y_train.iloc[val_idx]

    lgbm.fit(X_tr, Y_tr)
    oof[val_idx] = lgbm.predict(X_val)

    fold_rmse = root_mean_squared_error(Y_val, oof[val_idx])
    print(f"  Fold {fold+1} RMSE: {fold_rmse:.5f}")

oof_rmse = root_mean_squared_error(Y_train, oof)
print(f"\n✅ OOF RMSE : {oof_rmse:.5f}")

# ── Save ─────────────────────────────────────────────────────
if save: 
    np.save(f'../data/output/oof/oof_{save_name}.npy', oof)
    print(f"✅ OOF saved to ../data/output/oof/oof_{save_name}.npy")

  Fold 1 RMSE: 0.13751
  Fold 2 RMSE: 0.11772
  Fold 3 RMSE: 0.16555
  Fold 4 RMSE: 0.12415
  Fold 5 RMSE: 0.10437

✅ OOF RMSE : 0.13151
✅ OOF saved to ../data/output/oof/oof_round6_lightGBM_fe_trees.npy
